In [2]:
from pyspark.sql import functions as F
from data_generator import get_spark, generate_large_table, generate_small_lookup
import time
spark=get_spark('Case01_BroadcastJoin')

large_df=generate_large_table(spark,num_rows=20_000_00)
small_df=generate_small_lookup(spark,num_rows=5_000)

large_df.cache().count()
small_df.cache().count()

26/04/10 17:11:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


5000

# BAD: Default SortMergeJoin (shuffle both)

In [3]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [4]:
small_df_clean = small_df.drop("id")
start = time.time()
result_bad = large_df.join(small_df_clean, "category", "inner")
result_bad.write.mode("overwrite").parquet("/tmp/case01_bad")
print(f"SortMergeJoin: {time.time() - start:.1f}s")

26/04/10 17:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/10 17:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/10 17:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/10 17:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/10 17:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/10 17:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/10 17:12:05 WARN MemoryManager: Total allocation exceeds 95.

SortMergeJoin: 4.0s


# GOOD: Explicit Broadcast Join

In [5]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50MB")  

In [7]:
start = time.time()
result_good = large_df.join(F.broadcast(small_df_clean), "category", "inner")
result_good.write.mode("overwrite").parquet("/tmp/case01_good")
print(f" BroadcastJoin: {time.time() - start:.1f}s")

26/04/10 17:12:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/10 17:12:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/10 17:12:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/10 17:12:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/10 17:12:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/10 17:12:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/10 17:12:23 WARN MemoryManager: Total allocation exceeds 95.

 BroadcastJoin: 3.5s


In [ ]:
print("\n--- BAD PLAN ---")
result_bad.explain(True)
print("\n--- GOOD PLAN ---")
